In [38]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch
import json

In [40]:
# Load your JSON data
with open('questions.json', 'r') as f:
    data = json.load(f)


In [41]:
questions = [entry['question'] for entry in data]
answers = [entry['answer'] for entry in data]
labels = [0 if entry['in_syllabus'] else 1 for entry in data]

In [42]:
tokenizer = AutoTokenizer.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:157: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--bert-large-uncased-whole-word-masking-finetuned-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [43]:
model = AutoModelForQuestionAnswering.from_pretrained('bert-large-uncased-whole-word-masking-finetuned-squad')

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [52]:
def predict_answer(question):
    # Extract answers from the dataset
    contexts = [entry['answer'] for entry in data]  # Ensure 'data' contains your JSON dataset

    # Tokenize inputs
    inputs = tokenizer(question, contexts, return_tensors="pt", padding=True, truncation=True)

    # Forward pass, get logits
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the most likely answer
    answer_start_scores = outputs.start_logits
    answer_end_scores = outputs.end_logits

    # Get the tokens with the highest start and end scores
    answer_start = torch.argmax(answer_start_scores)
    answer_end = torch.argmax(answer_end_scores) + 1

    # Convert tokens to answer text
    answer = tokenizer.decode(inputs["input_ids"][0][answer_start:answer_end])

    return answer



In [53]:
question = "How can I add a customer?"
predicted_answer = predict_answer(question)


TypeError: TextInputSequence must be str

In [55]:
a = "Follow these steps:\n1. Download the Approval Form: Visit our website and download the approval form.\n2. Fill Out the Form: Complete all required fields in the form with accurate information.\n3. Send the Form: Email the filled-out form to the admin's email address provided on the website.\n4. Admin Approval: Wait for the admin to review and approve your form.\n5. Receive Email Confirmation: Once approved, you will receive an email notification.\n6. Create Your Account: Use the same email address you provided in the form to create your account on our website."

In [56]:
a.replace("\n","<br/>")

"Follow these steps:<br/>1. Download the Approval Form: Visit our website and download the approval form.<br/>2. Fill Out the Form: Complete all required fields in the form with accurate information.<br/>3. Send the Form: Email the filled-out form to the admin's email address provided on the website.<br/>4. Admin Approval: Wait for the admin to review and approve your form.<br/>5. Receive Email Confirmation: Once approved, you will receive an email notification.<br/>6. Create Your Account: Use the same email address you provided in the form to create your account on our website."